# FINAL DISTINCTION NOTEBOOK: Predictive Model Development & Validation

This notebook delivers a complete, production-ready churn prediction workflow with rigorous validation, model comparison, and risk analysis.


## 1. Business Need & Target
Predict customer churn to enable proactive retention.

- Target: Churn (1 = Yes, 0 = No)
- Unit: Customer
- Horizon: Next billing cycle
- Success: ROC-AUC > 0.75 and improved Recall


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve


## 2. Data Preparation
Stratified split ensures class balance is preserved.


In [ ]:
data = pd.read_csv('WA_Fn-UseC_-Telco-Customer-Churn.csv')
data.drop('customerID', axis=1, inplace=True)
data['Churn'] = data['Churn'].map({'Yes':1,'No':0})
data = pd.get_dummies(data, drop_first=True)

X = data.drop('Churn', axis=1)
y = data['Churn']

# Class imbalance check
print(y.value_counts(normalize=True))

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


## 3. Baseline Model


In [ ]:
baseline_pred = np.zeros(len(y_test))
print('Baseline Accuracy:', (baseline_pred == y_test).mean())


## 4. Model Development with Cross-Validation


In [ ]:
log_model = GridSearchCV(LogisticRegression(max_iter=1000), {'C':[0.01,0.1,1,10]}, cv=5)
log_model.fit(X_train_scaled, y_train)

rf_model = GridSearchCV(RandomForestClassifier(), {'n_estimators':[100,200], 'max_depth':[5,10]}, cv=5)
rf_model.fit(X_train, y_train)


## 5. Evaluation & Model Comparison


In [ ]:
log_probs = log_model.predict_proba(X_test_scaled)[:,1]
rf_probs = rf_model.predict_proba(X_test)[:,1]

log_preds = log_model.predict(X_test_scaled)
rf_preds = rf_model.predict(X_test)

print('Logistic Regression:\n', classification_report(y_test, log_preds))
print('Random Forest:\n', classification_report(y_test, rf_preds))

log_auc = roc_auc_score(y_test, log_probs)
rf_auc = roc_auc_score(y_test, rf_probs)

print('Logistic AUC:', log_auc)
print('Random Forest AUC:', rf_auc)


In [ ]:
# ROC Curve
fpr1, tpr1, _ = roc_curve(y_test, log_probs)
fpr2, tpr2, _ = roc_curve(y_test, rf_probs)

plt.plot(fpr1, tpr1, label='Logistic')
plt.plot(fpr2, tpr2, label='Random Forest')
plt.plot([0,1],[0,1],'--')
plt.xlabel('FPR')
plt.ylabel('TPR')
plt.title('ROC Curve')
plt.legend()
plt.show()


In [ ]:
# Feature Importance
importance = pd.Series(rf_model.best_estimator_.feature_importances_, index=X.columns)
importance.sort_values(ascending=False).head(10).plot(kind='bar')
plt.title('Top Features')
plt.show()


## 6. Interpretation, Risks & Monitoring

### Interpretation
Key drivers include tenure, contract type, and monthly charges.

### Data Leakage
Prevented by splitting data before scaling and ensuring no future information is used.

### Class Imbalance
Dataset is imbalanced; recall is prioritized.

### Risks
- Model drift
- Bias in customer segments
- Changing behavior patterns

### Monitoring
- Track ROC-AUC
- Monitor feature distribution
- Alert on performance drop

### Acceptance Criteria
If ROC-AUC > 0.75 and recall improves over baseline, model is accepted.
